# 🗄️ SQL Avanzado — LaLiga 2023/24
**Proyecto S2 | Sports Analytics Portfolio**  
Herramientas: Python · SQLite · Pandas  
Dataset: La Liga Stats 2023-2024

---
## Objetivos
Practicar SQL avanzado con datos reales de fútbol usando:
- **Window functions**: RANK, DENSE_RANK, NTILE, LAG, SUM OVER, AVG OVER PARTITION
- **CTEs** (Common Table Expressions)
- **Subqueries avanzadas**

## 0. Setup — Crear base de datos SQLite

In [1]:
import pandas as pd
import sqlite3
import os

os.makedirs('output', exist_ok=True)

# Cargar y limpiar dataset
df = pd.read_csv(r'C:\Users\jrubi\Desktop\Sports-Analytics Portfolio\S1 EDA La LIga\data\laliga_2324.csv')

# Una fila por jugador
df = df[df['competition'] == 'Classics'].reset_index(drop=True)

# Solo equipos LaLiga 23/24
equipos_laliga = [
    'Real Madrid', 'FC Barcelona', 'Atlético de Madrid', 'Girona FC',
    'Athletic Club', 'Real Sociedad', 'Real Betis', 'Villarreal CF',
    'Valencia CF', 'Getafe CF', 'Deportivo Alavés', 'CA Osasuna',
    'RC Celta', 'Rayo Vallecano', 'Sevilla FC', 'Cádiz CF',
    'UD Almería', 'Granada CF', 'UD Las Palmas', 'RCD Mallorca'
]
df = df[df['team'].isin(equipos_laliga)].reset_index(drop=True)

# Seleccionar columnas relevantes y renombrar para SQL
df_sql = df[[
    'name', 'team', 'position',
    'games_played', 'time_played',
    'goals', 'goal_assists',
    'total_shots', 'shots_on_target_inc_goals',
    'total_passes', 'total_successful_passes_excl_crosses_corners',
    'total_tackles', 'tackles_won',
    'successful_dribbles', 'interceptions',
    'yellow_cards', 'total_red_cards'
]].copy()

# Renombrar columnas para que sean más manejables en SQL
df_sql.columns = [
    'nombre', 'equipo', 'posicion',
    'partidos', 'minutos',
    'goles', 'asistencias',
    'tiros', 'tiros_puerta',
    'pases', 'pases_exitosos',
    'tackles', 'tackles_ganados',
    'regates', 'intercepciones',
    'amarillas', 'rojas'
]

# Rellenar nulos
df_sql = df_sql.fillna(0)

# Métricas calculadas
df_sql['minutos'] = df_sql['minutos'].astype(float)
df_sql['contribuciones'] = df_sql['goles'] + df_sql['asistencias']
df_sql['precision_tiro'] = (df_sql['tiros_puerta'] / df_sql['tiros'].replace(0, 1) * 100).round(1)
df_sql['goles_p90'] = (df_sql['goles'] / (df_sql['minutos'] / 90).replace(0, 1)).round(3)

# Crear base de datos SQLite en memoria
conn = sqlite3.connect(':memory:')
df_sql.to_sql('jugadores', conn, index=False, if_exists='replace')

print(f'✅ Base de datos creada: {len(df_sql)} jugadores · {df_sql["equipo"].nunique()} equipos')
print(f'   Tabla: jugadores')
print(f'   Columnas: {df_sql.columns.tolist()}')

✅ Base de datos creada: 637 jugadores · 20 equipos
   Tabla: jugadores
   Columnas: ['nombre', 'equipo', 'posicion', 'partidos', 'minutos', 'goles', 'asistencias', 'tiros', 'tiros_puerta', 'pases', 'pases_exitosos', 'tackles', 'tackles_ganados', 'regates', 'intercepciones', 'amarillas', 'rojas', 'contribuciones', 'precision_tiro', 'goles_p90']


In [3]:
# Función helper para ejecutar queries y ver resultados limpios
def query(sql, n=10):
    result = pd.read_sql_query(sql, conn)
    print(f'📊 {len(result)} filas devueltas — mostrando {min(n, len(result))}')
    return result.head(n)

print('✅ Helper query() listo — úsalo así: query("SELECT * FROM jugadores LIMIT 5")')

✅ Helper query() listo — úsalo así: query("SELECT * FROM jugadores LIMIT 5")


---
## Query 1 — RANK(): Ranking de goleadores por equipo

**Concepto**: `RANK() OVER (PARTITION BY equipo ORDER BY goles DESC)`  
→ Reinicia el ranking para cada equipo. Útil para ver quién es el máximo goleador de cada club.

In [7]:
query("""
SELECT
    nombre,
    equipo,
    goles,
    RANK() OVER (PARTITION BY equipo ORDER BY goles DESC) AS ranking_en_equipo
FROM jugadores
WHERE goles > 5
ORDER BY equipo, ranking_en_equipo
""", n=20)

📊 51 filas devueltas — mostrando 20


,nombre,equipo,goles,ranking_en_equipo
0,Gorka Guruzeta,Athletic Club,14.0,1
1,Iñaki Williams,Athletic Club,12.0,2
2,Alex Berenguer,Athletic Club,7.0,3
3,Antoine Griezmann,Atlético de Madrid,16.0,1
4,Álvaro Morata,Atlético de Madrid,15.0,2
5,Ángel Correa,Atlético de Madrid,9.0,3
6,Marcos Llorente,Atlético de Madrid,6.0,4
7,Ante Budimir,CA Osasuna,17.0,1
8,Raúl García,CA Osasuna,6.0,2
9,Samu Omorodion,Deportivo Alavés,8.0,1


In [9]:
# Versión útil: solo el máximo goleador de cada equipo
query("""
WITH ranking_equipo AS (
    SELECT
        nombre,
        equipo,
        goles,
        asistencias,
        RANK() OVER (PARTITION BY equipo ORDER BY goles DESC) AS rk
    FROM jugadores
)
SELECT nombre, equipo, goles, asistencias
FROM ranking_equipo
WHERE rk = 1
ORDER BY goles DESC
""", n=20)

📊 20 filas devueltas — mostrando 20


,nombre,equipo,goles,asistencias
0,Artem Dovbyk,Girona FC,24.0,8.0
1,Alexander Sørloth,Villarreal CF,23.0,6.0
2,Robert Lewandowski,FC Barcelona,19.0,8.0
3,Jude Bellingham,Real Madrid,19.0,6.0
4,Ante Budimir,CA Osasuna,17.0,2.0
5,Antoine Griezmann,Atlético de Madrid,16.0,6.0
6,Youssef En-Nesyri,Sevilla FC,16.0,2.0
7,Borja Mayoral,Getafe CF,15.0,1.0
8,Gorka Guruzeta,Athletic Club,14.0,5.0
9,Jørgen Strand Larsen,RC Celta,13.0,3.0


---
## Query 2 — DENSE_RANK(): Ranking global sin saltos

**Concepto**: `RANK()` salta números cuando hay empates (1,2,2,4). `DENSE_RANK()` no salta (1,2,2,3).  
→ Más justo para rankings donde los empates son frecuentes.

In [11]:
query("""
SELECT
    nombre,
    equipo,
    goles,
    RANK()       OVER (ORDER BY goles DESC) AS rank_con_saltos,
    DENSE_RANK() OVER (ORDER BY goles DESC) AS rank_sin_saltos
FROM jugadores
WHERE goles >= 5
ORDER BY goles DESC
""", n=20)

📊 65 filas devueltas — mostrando 20


,nombre,equipo,goles,rank_con_saltos,rank_sin_saltos
0,Artem Dovbyk,Girona FC,24.0,1,1
1,Alexander Sørloth,Villarreal CF,23.0,2,2
2,Jude Bellingham,Real Madrid,19.0,3,3
3,Robert Lewandowski,FC Barcelona,19.0,3,3
4,Ante Budimir,CA Osasuna,17.0,5,4
5,Antoine Griezmann,Atlético de Madrid,16.0,6,5
6,Youssef En-Nesyri,Sevilla FC,16.0,6,5
7,Borja Mayoral,Getafe CF,15.0,8,6
8,Vinícius José Paixão de Oliveira Júnior,Real Madrid,15.0,8,6
9,Álvaro Morata,Atlético de Madrid,15.0,8,6


---
## Query 3 — NTILE(): Clasificar jugadores en cuartiles

**Concepto**: `NTILE(4)` divide los jugadores en 4 grupos iguales.  
→ Cuartil 1 = top 25% goleadores, Cuartil 4 = bottom 25%.  
Muy usado en scouting para segmentar jugadores por rendimiento.

In [19]:
query("""
SELECT
    nombre,
    equipo,
    posicion,
    goles,
    contribuciones,
    NTILE(4) OVER (ORDER BY contribuciones DESC) AS cuartil
FROM jugadores
WHERE minutos >= 450
ORDER BY cuartil, contribuciones DESC
""", n=102)

📊 403 filas devueltas — mostrando 102


,nombre,equipo,posicion,goles,contribuciones,cuartil
0,Artem Dovbyk,Girona FC,Forward,24.0,32.0,1
1,Alexander Sørloth,Villarreal CF,Forward,23.0,29.0,1
2,Robert Lewandowski,FC Barcelona,Forward,19.0,27.0,1
3,Jude Bellingham,Real Madrid,Midfielder,19.0,25.0,1
4,Antoine Griezmann,Atlético de Madrid,Forward,16.0,22.0,1
...,...,...,...,...,...,...
97,Fran Pérez,Valencia CF,Midfielder,1.0,5.0,1
98,Gonçalo Manuel Ganchinho Guedes,Villarreal CF,Midfielder,3.0,5.0,1
99,Isi Palazón,Rayo Vallecano,Midfielder,4.0,5.0,1
100,Javi Guerra,Valencia CF,Midfielder,4.0,5.0,1


In [15]:
# Resumen por cuartil — ¿cómo de distinto es cada grupo?
query("""
WITH cuartiles AS (
    SELECT
        contribuciones,
        goles,
        NTILE(4) OVER (ORDER BY contribuciones DESC) AS cuartil
    FROM jugadores
    WHERE minutos >= 450
)
SELECT
    cuartil,
    COUNT(*) AS jugadores,
    ROUND(AVG(contribuciones), 2) AS media_contribuciones,
    ROUND(AVG(goles), 2)          AS media_goles,
    MAX(contribuciones)           AS max_contribuciones,
    MIN(contribuciones)           AS min_contribuciones
FROM cuartiles
GROUP BY cuartil
ORDER BY cuartil
""")

📊 4 filas devueltas — mostrando 4


,cuartil,jugadores,media_contribuciones,media_goles,max_contribuciones,min_contribuciones
0,1,101,10.64,6.62,32.0,5.0
1,2,101,3.45,1.75,5.0,2.0
2,3,101,1.47,0.66,2.0,1.0
3,4,100,0.16,0.07,1.0,0.0


---
## Query 4 — LAG(): Comparar con el jugador anterior en el ranking

**Concepto**: `LAG(goles, 1)` devuelve el valor de goles de la fila anterior.  
→ Permite calcular la diferencia entre posiciones consecutivas del ranking.  
Útil para ver dónde hay saltos grandes de rendimiento.

In [21]:
query("""
SELECT
    nombre,
    equipo,
    goles,
    LAG(goles, 1) OVER (ORDER BY goles DESC)                        AS goles_anterior,
    goles - LAG(goles, 1) OVER (ORDER BY goles DESC)                AS diferencia,
    RANK() OVER (ORDER BY goles DESC)                               AS ranking
FROM jugadores
WHERE goles >= 3
ORDER BY goles DESC
""", n=20)

📊 118 filas devueltas — mostrando 20


,nombre,equipo,goles,goles_anterior,diferencia,ranking
0,Artem Dovbyk,Girona FC,24.0,NaN,NaN,1
1,Alexander Sørloth,Villarreal CF,23.0,24.0,-1.0,2
2,Jude Bellingham,Real Madrid,19.0,23.0,-4.0,3
3,Robert Lewandowski,FC Barcelona,19.0,19.0,0.0,3
4,Ante Budimir,CA Osasuna,17.0,19.0,-2.0,5
5,Antoine Griezmann,Atlético de Madrid,16.0,17.0,-1.0,6
6,Youssef En-Nesyri,Sevilla FC,16.0,16.0,0.0,6
7,Borja Mayoral,Getafe CF,15.0,16.0,-1.0,8
8,Vinícius José Paixão de Oliveira Júnior,Real Madrid,15.0,15.0,0.0,8
9,Álvaro Morata,Atlético de Madrid,15.0,15.0,0.0,8


---
## Query 5 — SUM() OVER: Goles acumulados por equipo

**Concepto**: `SUM() OVER (PARTITION BY equipo ORDER BY goles DESC)`  
→ Acumula los goles conforme va sumando jugadores del equipo, ordenados de mayor a menor.  
Permite ver qué % de los goles aportan los top jugadores de cada equipo.

In [23]:
query("""
SELECT
    nombre,
    equipo,
    goles,
    SUM(goles) OVER (PARTITION BY equipo ORDER BY goles DESC) AS goles_acumulados_equipo,
    SUM(goles) OVER (PARTITION BY equipo)                     AS goles_totales_equipo,
    ROUND(
        100.0 * SUM(goles) OVER (PARTITION BY equipo ORDER BY goles DESC)
        / NULLIF(SUM(goles) OVER (PARTITION BY equipo), 0)
    , 1) AS pct_acumulado
FROM jugadores
WHERE equipo = 'Real Madrid'
  AND goles > 0
ORDER BY goles DESC
""")

📊 14 filas devueltas — mostrando 10


,nombre,equipo,goles,goles_acumulados_equipo,goles_totales_equipo,pct_acumulado
0,Jude Bellingham,Real Madrid,19.0,19.0,85.0,22.4
1,Vinícius José Paixão de Oliveira Júnior,Real Madrid,15.0,34.0,85.0,40.0
2,José Luis Mato Sanmartín,Real Madrid,10.0,54.0,85.0,63.5
3,Rodrygo Silva de Goes,Real Madrid,10.0,54.0,85.0,63.5
4,Brahim Díaz,Real Madrid,8.0,62.0,85.0,72.9
5,Arda Güler,Real Madrid,6.0,68.0,85.0,80.0
6,Dani Carvajal,Real Madrid,4.0,72.0,85.0,84.7
7,Aurélien Tchouaméni,Real Madrid,3.0,78.0,85.0,91.8
8,Lucas Vázquez,Real Madrid,3.0,78.0,85.0,91.8
9,Federico Valverde,Real Madrid,2.0,82.0,85.0,96.5


In [29]:
# ¿Cuántos jugadores aportan el 50% de los goles en cada equipo?
query("""
WITH acumulado AS (
    SELECT
        nombre,
        equipo,
        goles,
        ROUND(
            100.0 * SUM(goles) OVER (PARTITION BY equipo ORDER BY goles DESC)
            / NULLIF(SUM(goles) OVER (PARTITION BY equipo), 0)
        , 1) AS pct_acumulado
    FROM jugadores
    WHERE goles > 0
),
primeros_50 AS (
    SELECT equipo, COUNT(*) AS jugadores_para_50pct
    FROM acumulado
    WHERE pct_acumulado <= 50
    GROUP BY equipo
)
SELECT equipo, jugadores_para_50pct
FROM primeros_50
ORDER BY jugadores_para_50pct
""", n=20)

📊 20 filas devueltas — mostrando 20


,equipo,jugadores_para_50pct
0,CA Osasuna,1
1,Getafe CF,1
2,Granada CF,1
3,UD Almería,1
4,Valencia CF,1
5,Villarreal CF,1
6,Athletic Club,2
7,Atlético de Madrid,2
8,Cádiz CF,2
9,Deportivo Alavés,2


---
## Query 6 — AVG() OVER PARTITION: Media por posición como benchmark

**Concepto**: `AVG(goles) OVER (PARTITION BY posicion)`  
→ Calcula la media de goles de TODOS los jugadores de la misma posición.  
Permite comparar a cada jugador contra sus pares directos, no contra toda la liga.

In [39]:
query("""
SELECT
    nombre,
    equipo,
    posicion,
    goles,
    ROUND(AVG(goles) OVER (PARTITION BY posicion), 2)   AS media_posicion,
    ROUND(goles - AVG(goles) OVER (PARTITION BY posicion), 2) AS vs_media,
    CASE
        WHEN goles > AVG(goles) OVER (PARTITION BY posicion) THEN 'Por encima'
        WHEN goles < AVG(goles) OVER (PARTITION BY posicion) THEN 'Por debajo'
        ELSE 'En la media'
    END AS rendimiento
FROM jugadores
WHERE minutos >= 450
  AND posicion IN ('Forward', 'Midfielder')
ORDER BY posicion, vs_media DESC
""", n=25)

📊 235 filas devueltas — mostrando 25


,nombre,equipo,posicion,goles,media_posicion,vs_media,rendimiento
0,Artem Dovbyk,Girona FC,Forward,24.0,5.64,18.36,Por encima
1,Alexander Sørloth,Villarreal CF,Forward,23.0,5.64,17.36,Por encima
2,Robert Lewandowski,FC Barcelona,Forward,19.0,5.64,13.36,Por encima
3,Ante Budimir,CA Osasuna,Forward,17.0,5.64,11.36,Por encima
4,Antoine Griezmann,Atlético de Madrid,Forward,16.0,5.64,10.36,Por encima
5,Youssef En-Nesyri,Sevilla FC,Forward,16.0,5.64,10.36,Por encima
6,Borja Mayoral,Getafe CF,Forward,15.0,5.64,9.36,Por encima
7,Vinícius José Paixão de Oliveira Júnior,Real Madrid,Forward,15.0,5.64,9.36,Por encima
8,Álvaro Morata,Atlético de Madrid,Forward,15.0,5.64,9.36,Por encima
9,Gorka Guruzeta,Athletic Club,Forward,14.0,5.64,8.36,Por encima


---
## Query 7 — CTE: Jugadores que superan la media de su posición

**Concepto**: Un CTE (WITH) permite definir una tabla temporal y reutilizarla.  
→ Más legible que subqueries anidadas. Estándar en analytics profesional.

In [41]:
query("""
WITH media_por_posicion AS (
    -- Paso 1: calcular media de contribuciones por posición
    SELECT
        posicion,
        ROUND(AVG(contribuciones), 2) AS media_contribuciones,
        ROUND(AVG(goles_p90), 3)      AS media_goles_p90,
        COUNT(*)                       AS total_jugadores
    FROM jugadores
    WHERE minutos >= 450
    GROUP BY posicion
),
jugadores_destacados AS (
    -- Paso 2: filtrar jugadores que superan la media de su posición
    SELECT
        j.nombre,
        j.equipo,
        j.posicion,
        j.goles,
        j.asistencias,
        j.contribuciones,
        j.goles_p90,
        m.media_contribuciones,
        ROUND(j.contribuciones - m.media_contribuciones, 2) AS diferencia_vs_media
    FROM jugadores j
    JOIN media_por_posicion m ON j.posicion = m.posicion
    WHERE j.minutos >= 450
      AND j.contribuciones > m.media_contribuciones
)
-- Paso 3: mostrar top por posición
SELECT *
FROM jugadores_destacados
ORDER BY diferencia_vs_media DESC
""", n=20)

📊 155 filas devueltas — mostrando 20


,nombre,equipo,posicion,goles,asistencias,contribuciones,goles_p90,media_contribuciones,diferencia_vs_media
0,Artem Dovbyk,Girona FC,Forward,24.0,8.0,32.0,0.829,8.23,23.77
1,Jude Bellingham,Real Madrid,Midfielder,19.0,6.0,25.0,0.736,3.96,21.04
2,Alexander Sørloth,Villarreal CF,Forward,23.0,6.0,29.0,0.831,8.23,20.77
3,Robert Lewandowski,FC Barcelona,Forward,19.0,8.0,27.0,0.620,8.23,18.77
4,Antoine Griezmann,Atlético de Madrid,Forward,16.0,6.0,22.0,0.543,8.23,13.77
5,Álex Baena,Villarreal CF,Midfielder,2.0,14.0,16.0,0.069,3.96,12.04
6,Vinícius José Paixão de Oliveira Júnior,Real Madrid,Forward,15.0,5.0,20.0,0.721,8.23,11.77
7,Ante Budimir,CA Osasuna,Forward,17.0,2.0,19.0,0.625,8.23,10.77
8,Gorka Guruzeta,Athletic Club,Forward,14.0,5.0,19.0,0.562,8.23,10.77
9,Iago Aspas,RC Celta,Forward,9.0,10.0,19.0,0.298,8.23,10.77


---
## Query 8 — Subquery avanzada: Equipos más eficientes vs media de liga

**Concepto**: Subquery en el FROM para calcular métricas de equipo y compararlas con la media global.  
→ Identifica qué equipos convierten mejor sus oportunidades respecto al resto de la liga.

In [43]:
query("""
WITH stats_equipo AS (
    -- Métricas agregadas por equipo
    SELECT
        equipo,
        SUM(goles)        AS goles_totales,
        SUM(tiros)        AS tiros_totales,
        SUM(tiros_puerta) AS tiros_puerta_totales,
        SUM(asistencias)  AS asistencias_totales,
        COUNT(DISTINCT nombre) AS jugadores
    FROM jugadores
    GROUP BY equipo
),
eficiencia AS (
    -- Calcular ratios de eficiencia
    SELECT
        equipo,
        goles_totales,
        tiros_totales,
        ROUND(100.0 * goles_totales / NULLIF(tiros_totales, 0), 1)        AS conversion_pct,
        ROUND(100.0 * tiros_puerta_totales / NULLIF(tiros_totales, 0), 1) AS precision_pct
    FROM stats_equipo
    WHERE tiros_totales > 0
),
media_liga AS (
    -- Media de conversión de toda la liga
    SELECT
        ROUND(AVG(conversion_pct), 2) AS media_conversion,
        ROUND(AVG(precision_pct), 2)  AS media_precision
    FROM eficiencia
)
SELECT
    e.equipo,
    e.goles_totales,
    e.tiros_totales,
    e.conversion_pct,
    e.precision_pct,
    m.media_conversion,
    ROUND(e.conversion_pct - m.media_conversion, 1) AS vs_media_liga,
    CASE
        WHEN e.conversion_pct > m.media_conversion THEN '✅ Por encima'
        ELSE '❌ Por debajo'
    END AS eficiencia_relativa
FROM eficiencia e, media_liga m
ORDER BY e.conversion_pct DESC
""", n=20)

📊 20 filas devueltas — mostrando 20


,equipo,goles_totales,tiros_totales,conversion_pct,precision_pct,media_conversion,vs_media_liga,eficiencia_relativa
0,Girona FC,84.0,380.0,22.1,50.5,13.82,8.3,✅ Por encima
1,Villarreal CF,62.0,319.0,19.4,50.5,13.82,5.6,✅ Por encima
2,Real Madrid,85.0,448.0,19.0,55.6,13.82,5.2,✅ Por encima
3,Atlético de Madrid,68.0,373.0,18.2,54.7,13.82,4.4,✅ Por encima
4,FC Barcelona,76.0,429.0,17.7,53.6,13.82,3.9,✅ Por encima
5,Athletic Club,59.0,348.0,17.0,49.1,13.82,3.2,✅ Por encima
6,CA Osasuna,43.0,299.0,14.4,41.1,13.82,0.6,✅ Por encima
7,Real Betis,46.0,323.0,14.2,47.1,13.82,0.4,✅ Por encima
8,Real Sociedad,48.0,344.0,14.0,45.1,13.82,0.2,✅ Por encima
9,Valencia CF,39.0,283.0,13.8,47.0,13.82,-0.0,❌ Por debajo


---
## Resumen de Window Functions aprendidas

| Función | Uso en fútbol |
|---|---|
| `RANK()` | Ranking de goleadores por equipo |
| `DENSE_RANK()` | Ranking global sin saltos por empates |
| `NTILE(4)` | Segmentar jugadores en cuartiles de rendimiento |
| `LAG()` | Diferencia entre posiciones consecutivas del ranking |
| `SUM() OVER` | Goles acumulados y % de contribución por equipo |
| `AVG() OVER PARTITION` | Benchmark de cada jugador vs media de su posición |
| CTE (WITH) | Análisis en pasos legibles y reutilizables |
| Subquery avanzada | Comparar equipos contra media de liga |

---
*Proyecto por: Jrubiols | ADE Bilingüe + Data Analytics | Sports Analytics Portfolio*  
*Dataset: La Liga Stats 2023-2024 — Kaggle*